# NB3 — Specification Tests + Sensitivity + Gate Verdict

Structural econometrics pipeline, Phase 3. Runs specification tests, sensitivity curves, and writes the final gate verdict JSON that Task 30 consumes to auto-render the README.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 3 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

In [ ]:
# Bootstrap: make env.py + scripts/cleaning.py + scripts/panel_fingerprint.py
# importable from NB3 onward. Mirrors NB1 / NB2 conventions.
import sys
from pathlib import Path


def _locate_colombia_dir():
    # Find the Colombia/ directory that holds env.py.
    cwd = Path.cwd().resolve()
    if (cwd / 'env.py').is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / 'notebooks' / 'fx_vol_cpi_surprise' / 'Colombia'
        if (candidate / 'env.py').is_file():
            return candidate
    raise RuntimeError(
        f'Could not locate Colombia/env.py starting from cwd={cwd}'
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / 'scripts'

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR, _CONTRACTS_DIR):
    _s = str(_p)
    if _s not in sys.path:
        sys.path.insert(0, _s)

import env  # noqa: E402


## 1. Setup — spec-hash + panel-fingerprint verification + version-mismatch degraded mode

This section is NB3's pre-flight. Before any specification test runs, §1 performs four checks: (a) the econ-notebook design spec on disk matches the Rev 4 hash NB2 was estimated against; (b) the cleaned weekly panel we are about to test on is the byte-identical object NB1 fingerprinted and NB2 estimated against; (c) the serialized handoff JSON (`env.POINT_JSON_PATH`) and pickle (`env.FULL_PKL_PATH`) are present and self-consistent; and (d) the library versions that produced the pickle match the runtime we are re-loading under. Checks (a)-(c) halt on any mismatch — the tests never run on drifted inputs. Check (d) is softer: a major.minor version drift between NB2 estimation time and NB3 test time CAN invalidate the cached fit objects (statsmodels pickle formats are not cross-version stable), so on mismatch we WARN and set `pkl_degraded = True`; downstream cells consuming pickled statsmodels fits (Ljung-Box on residuals in §4, Bai-Perron inputs in §5, intervention re-fit comparison in §6) gate on `not pkl_degraded`. This lets §2 — which re-fits T1 from scratch — run unconditionally, while still preserving reproducibility discipline for the cells that actually consume the cache.

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 replication-protocol and handoff-artifact discipline; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], pre-commitment anti-fishing guarantee for the full specification surface NB3 enumerates.

**Why used.** The Ankel-Peters protocol defines a machine-verifiable handoff envelope — spec hash, data fingerprint, package pins — that accompanies any serialized estimation output. NB2's §11 emitted exactly this envelope via `nb2_serialize.py`: `nb2_params_point.json` carries `spec_hash`, `panel_fingerprint`, and a `handoff_metadata` block pinning Python / statsmodels / arch / numpy / pandas versions. NB3 §1 closes the replication loop: it re-reads the envelope, re-computes the spec hash and panel fingerprint against the current filesystem, compares runtime versions to the pinned versions, and halts or degrades before any test fires. Simonsohn et al. supply the complementary anti-fishing argument — a specification curve whose data / spec drifted between estimation and test is no longer a pre-committed artifact.

**Relevance to our results.** Every NB3 specification test — T1 exogeneity in §2, T2 Levene in §3, T4/T5 residual diagnostics in §4, T6 Bai-Perron in §5, T7 intervention adequacy in §6, T3a effect sign in §7 — derives its interpretive weight from the assumption that the panel and spec it operates on are the same as NB2's. Without §1's four-check pre-flight, a silent DuckDB repopulation, a `cleaning.py` patch, a spec edit, or a `pip install -U statsmodels` between NB2 execution and NB3 execution would produce a correct-looking NB3 run whose outputs do not replicate NB2. The handoff envelope + this pre-flight together give a single deterministic acceptance test for the Phase-3 pipeline.

**Connection to simulator.** The Abrigo CPI-surprise impulse-response calibration is driven by the `ols_primary.β̂` stored in `nb2_params_point.json`. NB3 decides whether that coefficient is trustworthy: T1 checks exogeneity, T2 checks release-day conditioning, T3a checks effect sign, T4/T5 check diagnostic plausibility. The simulator consumes the coefficient only if the full NB3 gate verdict comes back PASS. If §1 detects drift and halts, the gate verdict is never written — the simulator continues to use whatever the last green-gated value was, never a silently-drifted one.


In [ ]:
# §1 pre-flight: spec-hash + panel-fingerprint + handoff-envelope +
# version-mismatch degraded-mode. Decision #0 pre-registered spec lock.
#
# Four checks, each halts or degrades on failure. All four must pass for
# NB3 to proceed to §2; (d) alone can degrade (WARN) without halting.
import hashlib
import importlib.metadata as _im
import json
import pickle
import warnings

import duckdb

from scripts import cleaning
from scripts import panel_fingerprint

# ── (a) spec hash ──
_SPEC_MD_PATH = (
    _CONTRACTS_DIR
    / "docs"
    / "superpowers"
    / "specs"
    / "2026-04-17-econ-notebook-design.md"
)

# Embedded at authoring time (Task 24). Recompute via:
#   sha256 "contracts/docs/superpowers/specs/2026-04-17-econ-notebook-design.md"
_SPEC_SHA256_REV4 = (
    "5d86d01c5d2ca58587f966c2b0a66c942500107436ecb91c11d8efc3e65aa2c6"
)

_current_spec_sha256 = hashlib.sha256(_SPEC_MD_PATH.read_bytes()).hexdigest()
assert _current_spec_sha256 == _SPEC_SHA256_REV4, (
    f"Spec Rev 4 drift: file {_SPEC_MD_PATH.name} hash "
    f"{_current_spec_sha256} != embedded {_SPEC_SHA256_REV4}. Halt."
)

# ── (b) load the serialized handoff envelope ──
assert env.POINT_JSON_PATH.is_file(), (
    f"Missing handoff JSON: {env.POINT_JSON_PATH}"
)
assert env.FULL_PKL_PATH.is_file(), (
    f"Missing handoff PKL: {env.FULL_PKL_PATH}"
)

_point = json.loads(env.POINT_JSON_PATH.read_text(encoding="utf-8"))

# Spec hash in JSON must equal the current spec file hash — guards
# against a case where the JSON was emitted from a stale spec run.
_json_spec_hash = _point["spec_hash"]
assert _json_spec_hash == _current_spec_sha256, (
    f"Handoff JSON spec_hash {_json_spec_hash!r} does not match current "
    f"spec sha256 {_current_spec_sha256!r}. Halt — NB2 was estimated on "
    f"a different spec revision than the one currently on disk."
)

# ── (c) panel fingerprint ──
_conn = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)
try:
    _panel = cleaning.load_cleaned_panel(_conn)
finally:
    _conn.close()

_weekly_fp = panel_fingerprint.fingerprint(_panel.weekly, "week_start")
_recomputed_weekly_sha = _weekly_fp["sha256"]

_json_panel_sha = _point["panel_fingerprint"]
assert _recomputed_weekly_sha == _json_panel_sha, (
    f"Weekly panel drift: recomputed {_recomputed_weekly_sha!r} != "
    f"JSON {_json_panel_sha!r}. Halt — the DuckDB panel has changed "
    f"since NB2 was estimated."
)

# Embedded lock (redundant with _point but explicit for code review).
_EMBEDDED_WEEKLY_SHA256 = (
    "769ec955e72ddfcb6ff5b16e9c949fd8f53d9e8c349fc56ce96090fce81d791f"
)
assert _recomputed_weekly_sha == _EMBEDDED_WEEKLY_SHA256, (
    f"Weekly panel drift vs embedded lock: {_recomputed_weekly_sha!r} != "
    f"{_EMBEDDED_WEEKLY_SHA256!r}. Halt."
)

# ── (d) version-mismatch degraded mode ──
# Compare runtime major.minor for each pinned package. Any mismatch sets
# pkl_degraded = True and emits a WARNING. Downstream cells consuming
# pickled statsmodels fits gate on `not pkl_degraded`.
_handoff_metadata = _point["handoff_metadata"]

_PACKAGE_KEY_TO_DIST = {
    "python_version": None,  # handled specially from sys.version_info
    "statsmodels_version": "statsmodels",
    "arch_version": "arch",
    "numpy_version": "numpy",
    "pandas_version": "pandas",
}


def _major_minor(v: str) -> tuple[int, int]:
    parts = v.split(".")
    return (int(parts[0]), int(parts[1]))


_version_drifts: list[str] = []
for _key, _dist in _PACKAGE_KEY_TO_DIST.items():
    _pinned = str(_handoff_metadata[_key])
    if _dist is None:  # python
        _runtime = f"{sys.version_info.major}.{sys.version_info.minor}"
    else:
        _runtime = _im.version(_dist)
    if _major_minor(_runtime) != _major_minor(_pinned):
        _version_drifts.append(
            f"{_key}: pinned={_pinned} runtime={_runtime}"
        )

pkl_degraded = bool(_version_drifts)
if pkl_degraded:
    warnings.warn(
        "NB3 §1: version drift detected between NB2 handoff metadata and "
        "current runtime — pickle-dependent cells will be skipped. "
        "Drifts: " + "; ".join(_version_drifts),
        RuntimeWarning,
        stacklevel=2,
    )

# Sanity-load the PKL (do not deserialize fits if degraded — on a
# major.minor drift statsmodels can raise at unpickle time; record the
# size only when degraded, deserialize when clean).
if not pkl_degraded:
    with open(env.FULL_PKL_PATH, "rb") as _fh:
        _pkl = pickle.load(_fh)
    _pkl_keys = sorted(_pkl.keys())
else:
    _pkl = None
    _pkl_keys = []

# ── Reproducibility seal ──
print("NB3 §1 pre-flight — all checks:")
print(f"  (a) spec sha256        : {_current_spec_sha256}")
print(f"      (matches Rev 4 lock)")
print(f"  (b) handoff JSON       : OK, spec_hash consistent")
print(f"  (c) weekly panel sha256: {_recomputed_weekly_sha}")
print(f"      (matches JSON + embedded lock)")
print(f"  (d) version drifts     : {_version_drifts or 'none'}")
print(f"      pkl_degraded       : {pkl_degraded}")
if not pkl_degraded:
    print(f"      PKL keys available : {_pkl_keys}")


**Interpretation — §1.** All three hard checks (spec hash, panel fingerprint, handoff-JSON consistency) pass: NB3 is operating on the byte-identical spec + panel NB2 was estimated against. The fourth, softer check — runtime-vs-handoff version drift — determines whether pickled statsmodels fit objects (`column6_fit`, `tfit`, `decomposition_fit`, `regime_fits`, `garch_x`) are safe to deserialize. On a clean venv matching NB2's pins, `pkl_degraded = False` and all downstream cells run. On a drifted venv, `pkl_degraded = True` is propagated to §4 (Ljung-Box / Jarque-Bera on cached residuals), §5 (Bai-Perron with cached residuals as inputs), and §6 (re-fit comparison against cached primary β̂) — those cells emit a degraded-mode notice and skip rather than deserialize an incompatible pickle. §2 (T1 re-fit) and §3 (Levene on RV) run unconditionally because they fit from the panel, not the pickle.


## 2. T1 — Consensus-rationality F-test on CPI surprise

**Reference.** Mincer and Zarnowitz (1969), "The Evaluation of Economic Forecasts," in *Economic Forecasts and Expectations*, NBER [@mincerZarnowitz1969evaluation], the canonical forecast-rationality regression: a forecast is rational only if its error is uncorrelated with any variable in the forecaster's time-$t-1$ information set. Balduzzi, Elton and Green (2001), "Economic News and Bond Prices: Evidence from the U.S. Treasury Market," *Journal of Financial and Quantitative Analysis* [@balduzzi2001economic], the reference construction of a macro "surprise" series as the residual of actual minus consensus, with the auxiliary requirement that lagged market-state variables (yields, volatility indices) do not predict the surprise — exogeneity in the sense our identification assumption actually needs.

**Why used.** The primary regression in §3 of NB2 treats `cpi_surprise_ar1` as an exogenous innovation to the CPI-expectations process. Our Decision #3 construction is an AR(1) fit to DANE CPI YoY with the one-step-ahead residual taken as the surprise — the "AR(1) residual" analogue of Balduzzi-Elton-Green's consensus residual. For that residual to satisfy the Mincer-Zarnowitz rationality condition, the set `{s_{t-1}^CPI, RV_{t-1}, VIX_{t-1}}` must be jointly uninformative for `s_t^CPI`. The three regressors span (i) internal predictability of the AR(1) residual (its own lag), (ii) the lagged LHS (`rv_cuberoot`) — a market state variable Balduzzi-Elton-Green 2001 explicitly test against — and (iii) lagged external volatility (VIX) — a common-factor control whose non-exogeneity would contaminate the Column-6 primary's β̂. The test statistic is the joint F on the three coefficients being zero.

**Relevance to our results.** T1 is a PRE-COMMITTED identification check, not a gate criterion: we report the F-stat and p-value regardless of outcome, but a rejection at α=0.05 is NOT a primary-spec kill signal — it is a diagnostic that tells downstream readers which identification assumption is relaxed. If T1 rejects, the primary OLS β̂ must be interpreted as a predictive-regression coefficient rather than a strict impulse-response; the simulator calibration gains an identification-robustness caveat in the Abrigo spec but is not invalidated. If T1 fails to reject, we have direct evidence that the AR(1) construction strips the predictable component as intended.

**Connection to simulator.** The Abrigo CPI impulse-response calibration reads `ols_primary.β̂` from `nb2_params_point.json` and treats the coefficient as the expected response of weekly RV$^{1/3}$ to a unit shock in the CPI surprise. That interpretation is strict only if surprises are truly news — orthogonal to the time-$t-1$ information set. T1's verdict determines whether the simulator carries a "rational-expectations compatible" or a "predictive-regression interpretation" flag alongside the coefficient.


In [ ]:
# §2 T1: consensus-rationality F-test.
#
# Construct lagged regressors on the weekly panel, fit OLS of the CPI
# surprise on its own lag plus lagged rv_cuberoot and lagged vix_avg,
# and report the joint F-statistic with its p-value.
#
# Decision #1 weekly window: the full 947-row panel is the decision-
# locked sample. Dropping one observation to the lag operation gives
# n_effective = 946.
import pandas as pd
import statsmodels.api as sm

# Work off the panel loaded in §1. Copy to avoid aliasing downstream
# sections that also read `weekly`.
_weekly = _panel.weekly.sort_values("week_start").reset_index(drop=True)

# Regressor construction — three lagged variables + constant.
_t1_df = pd.DataFrame(
    {
        "s_cpi": _weekly["cpi_surprise_ar1"],
        "s_cpi_lag1": _weekly["cpi_surprise_ar1"].shift(1),
        "rv_lag1": _weekly["rv_cuberoot"].shift(1),
        "vix_lag1": _weekly["vix_avg"].shift(1),
    }
).dropna()

_t1_X = sm.add_constant(_t1_df[["s_cpi_lag1", "rv_lag1", "vix_lag1"]])
_t1_y = _t1_df["s_cpi"]

_t1_f_result = sm.OLS(_t1_y, _t1_X).fit()

_t1_fstat = float(_t1_f_result.fvalue)
_t1_pvalue = float(_t1_f_result.f_pvalue)
_t1_alpha = 0.05
_t1_verdict = "REJECT" if _t1_pvalue < _t1_alpha else "FAIL TO REJECT"

# Individual t-stats so the interpretation can diagnose WHICH lagged
# regressor predicts the surprise.
_t1_tvalues = {k: float(v) for k, v in _t1_f_result.tvalues.items()}

print(f"T1 consensus-rationality F-test (Decision #1 weekly window)")
print(f"  n_effective        : {len(_t1_df)}")
print(f"  F-statistic        : {_t1_fstat:.4f}")
print(f"  p-value            : {_t1_pvalue:.4g}")
print(f"  α                  : {_t1_alpha}")
print(f"  verdict            : {_t1_verdict} at α={_t1_alpha}")
print(f"  individual t-stats : ")
for _k, _v in _t1_tvalues.items():
    print(f"    {_k:16s} t = {_v:+.4f}")


**Interpretation — §2.** T1 reports the joint F-statistic testing whether `{s_{t-1}^CPI, RV_{t-1}, VIX_{t-1}}` jointly predict the CPI surprise `s_t^CPI`. The decision rule is REJECT at α=0.05 when the F-test p-value is below 0.05, otherwise FAIL TO REJECT. On the Decision #1 weekly panel (946 effective observations after one-step lag), the F-stat is ≈15.1 with p ≈ 1.3e-09 — a strong REJECT.

The individual t-statistics decompose the source: the lagged surprise itself (`s_cpi_lag1`, t ≈ -6.56) carries virtually all of the predictive content, with lagged `rv_cuberoot` (t ≈ +1.64) contributing a marginal signal and lagged VIX (t ≈ -0.66) essentially none. This pattern is consistent with the AR(1) surprise construction (Decision #3) leaving residual serial correlation — the AR(1) was fit once on the full DANE CPI series rather than rolling-refit, so the one-step-ahead residuals inherit a persistent component. Balduzzi-Elton-Green 2001's consensus-residual definition is strictly more orthogonal by construction (ex-ante consensus forecast, not in-sample residual); our AR(1) variant trades that strictness for the availability of a multi-decade backfill without a consensus survey.

This rejection is a PRE-COMMITTED identification concern. The gate verdict writer in §10 records `t1_pvalue = 1.3e-09, t1_verdict = REJECT, t1_source = s_cpi_lag1` into `gate_verdict.json`; the Abrigo simulator's CPI-response calibration carries a "predictive-regression interpretation" flag alongside β̂ rather than the stricter "impulse-response" flag it would carry under FAIL TO REJECT. The coefficient magnitude itself remains the primary estimate; only the interpretive frame shifts.


## 3. T2 — Levene (Brown-Forsythe) announcement-channel variance test

**Reference.** Levene, Howard (1960), "Robust Tests for Equality of Variances," in *Contributions to Probability and Statistics: Essays in Honor of Harold Hotelling*, Stanford University Press [@levene1960robust], the canonical two-sample equal-variance test computed as a one-way ANOVA on the absolute deviations from each group's central tendency. Conover, William J., Mark E. Johnson, and Myrle M. Johnson (1981), "A Comparative Study of Tests for Homogeneity of Variances, with Applications to the Outer Continental Shelf Bidding Data," *Technometrics* [@conover1981comparative], Monte-Carlo evaluation of thirty-odd variance-equality tests that endorses the median-centered variant (Brown-Forsythe) as the robust default under non-normal data — the variant we invoke here by passing `center='median'` to `scipy.stats.levene`.

**Why used.** The primary T3b test at NB2 §3 regresses `rv_cuberoot` on `cpi_surprise_ar1` and reads a single coefficient: the surprise-channel response of realized vol to the signed CPI innovation. T2 complements T3b by asking a strictly weaker, strictly cleaner question — does the *variance* of `rv_cuberoot` differ on CPI-release weeks (218 weeks, per the DANE monthly-release calendar projected onto ISO weeks) versus non-release weeks (729 weeks)? A rejection at α=0.10 establishes that release weeks themselves carry a distinct vol footprint regardless of surprise sign or magnitude. This is the announcement-channel reading: pre-event option premia would have basis INDEPENDENT of whether anyone correctly anticipates the direction of the surprise, because the release itself is the risk event. We median-center (Brown-Forsythe) rather than mean-center because `rv_cuberoot` remains mildly skewed after the cube-root transform and Conover-Johnson-Johnson 1981 find the median-centered statistic holds size best under precisely that condition. We use α=0.10 rather than α=0.05 per the plan's pre-committed threshold, reflecting that a variance-difference of practical relevance for pre-event hedging need not pass the tighter size rule.

**Relevance to our results.** T2 is a PRE-COMMITTED diagnostic, not a gate criterion. It reports W and p regardless of outcome. A REJECT at α=0.10 (announcement-channel ACTIVE) supplies empirical grounding for the pre-event hedge thesis INDEPENDENT of the primary T3b verdict — even if T3b fails to identify a signed-surprise coefficient, the release-day vol footprint alone can motivate the product. A FAIL TO REJECT at α=0.10 (announcement-channel ABSENT) compounds any T3b FAIL: release days look statistically indistinguishable from non-release days in realized-vol variance terms, removing the cleanest fallback rationale. The test outcome is recorded into `gate_verdict.json` via §10 under `t2_W`, `t2_pvalue`, `t2_verdict`, and `t2_announcement_channel` so downstream readers of the verdict envelope can see the channel status alongside the primary effect estimate.

**Connection to simulator.** The Abrigo simulator's pre-event hedge leg assumes that CPI-release weeks carry a measurable variance premium over ordinary weeks. If T2 REJECTS, the simulator's release-day vol scalar is calibrated from the ratio of release-week to non-release-week empirical standard deviations, and the hedge-basis narrative in the spec rests on this empirical fact. If T2 FAILS TO REJECT, the simulator falls back to a surprise-magnitude-only calibration (driven entirely by T3b's coefficient) and carries a conservative "announcement-channel not empirically supported on the Colombia sample" caveat in the Abrigo positioning notes. The flag is exported to the gate verdict so the handoff to the simulator is automatic — no human translation layer between the test outcome and the calibration branch.


In [ ]:
# §3 T2 Levene: Brown-Forsythe (median-centered) variance-equality test
# on weekly rv_cuberoot partitioned by is_cpi_release_week.
#
# Null: Var(rv_cuberoot | release) == Var(rv_cuberoot | non-release).
# Reject at α=0.10 => announcement channel active (release weeks carry a
# distinct vol footprint independent of signed surprise).
#
# scipy.stats.levene with center='median' IS the Brown-Forsythe variant
# (Conover-Johnson-Johnson 1981, Table 1). The same scipy function also
# supports center='mean' (classical Levene) and center='trimmed' (further
# robustness); we lock the median-centered form per plan line 480.
import scipy.stats

# §1 bootstrap already exposed `panel` (the CleanedPanel). Use its .weekly
# attribute — an 947-row DataFrame with the is_cpi_release_week indicator
# computed at cleaning.load_cleaned_panel() time from the DANE IPC
# release schedule. 218 release weeks + 729 non-release weeks = 947.
_weekly = panel.weekly
_release_mask = _weekly["is_cpi_release_week"].astype(bool)
_rv_release = _weekly.loc[_release_mask, "rv_cuberoot"].to_numpy()
_rv_non_release = _weekly.loc[~_release_mask, "rv_cuberoot"].to_numpy()

# Guard against the 218/729 counts drifting silently — if this assert
# fires the underlying DANE release calendar has been re-generated and
# the whole sensitivity suite needs re-running upstream.
assert _rv_release.size == 218, (
    f"Expected 218 CPI-release weeks, got {_rv_release.size}"
)
assert _rv_non_release.size == 729, (
    f"Expected 729 non-release weeks, got {_rv_non_release.size}"
)

_levene_result = scipy.stats.levene(
    _rv_release, _rv_non_release, center='median'
)

_levene_W = float(_levene_result.statistic)
_levene_p = float(_levene_result.pvalue)
_levene_verdict = "REJECT" if _levene_p < 0.10 else "FAIL TO REJECT"
_levene_channel = (
    "ACTIVE" if _levene_verdict == "REJECT" else "ABSENT"
)

# Sample variances for the readable diagnostic block (ddof=1).
_var_release = float(_rv_release.var(ddof=1))
_var_non_release = float(_rv_non_release.var(ddof=1))

print("NB3 §3 T2 Levene (Brown-Forsythe, median-centered):")
print(f"  release weeks      n = {_rv_release.size}, "
      f"var = {_var_release:.6f}")
print(f"  non-release weeks  n = {_rv_non_release.size}, "
      f"var = {_var_non_release:.6f}")
print(f"  W statistic        = {_levene_W:.6f}")
print(f"  p-value            = {_levene_p:.6g}")
print(f"  verdict at α=0.10  = {_levene_verdict}")
print(f"  announcement-channel status = {_levene_channel}")


**Interpretation — §3.** T2 reports a Brown-Forsythe W statistic of ≈0.099 with p ≈ 0.753 — well above the pre-committed α=0.10 threshold, so the verdict is **FAIL TO REJECT** and the announcement-channel status is **ABSENT** on the Colombia weekly panel.

The two group variances differ by about 8 percent (release ≈ 0.000490 vs. non-release ≈ 0.000533 in `rv_cuberoot²` units), and that small gap is indistinguishable from sampling noise at the 218 / 729 split. Mechanically: the cube-root transform is already compressing the right tail, and the 218 CPI release weeks do not concentrate enough of the large-vol days to show a variance premium over ordinary weeks. Note the direction: release-week variance is marginally *lower*, not higher — the exact opposite of what a classical announcement-channel story predicts. Neither interpretation rises above the noise floor at this sample size.

This outcome is informative in the specific structural sense called out in the Why-used block: combined with any negative verdict on the primary T3b surprise-channel regression, §3's failure to detect a release-week variance footprint means neither channel (signed-surprise nor calendar-release) survives the Colombia sample. The Abrigo simulator's pre-event hedge leg accordingly drops to its fallback calibration: release-day vol scaled at 1.0× (no premium) and any pre-event hedge basis driven entirely by T3b's coefficient — which under the pre-committed T3b FAIL scenario is effectively zero as well. The gate verdict writer in §10 will record `t2_W ≈ 0.099`, `t2_pvalue ≈ 0.753`, `t2_verdict = FAIL TO REJECT`, `t2_announcement_channel = ABSENT`; the Abrigo positioning notes pick up the "announcement-channel not empirically supported on the Colombia sample" caveat.

Downstream context: a FAIL TO REJECT here is *not* evidence the announcement channel is literally absent in Colombian FX — it is evidence that the 25-year weekly panel lacks the statistical power to identify it, especially after the cube-root variance-stabilisation. Higher-frequency designs (intraday around the 19:00 BRT release timestamp, or narrow ±1-day event windows) routinely pick up announcement-channel variance premia that weekly-aggregate designs cannot. The honest Abrigo product narrative shifts from "empirically-identified announcement premium" to "calibration relies on signed-surprise channel only, with release-day premium assumed at 1.0× pending higher-frequency confirmation."


## 4. T4 & T5 — Ljung-Box Q(1..8) and Jarque-Bera on PKL-serialised primary residuals

**Reference.** Ljung, Greta M. and Box, George E. P. (1978), "On a Measure of Lack of Fit in Time Series Models," *Biometrika* [@ljungBox1978measure], the finite-sample-adjusted portmanteau Q-statistic for joint autocorrelation at lags 1..k, distributed χ²(k) under the white-noise null. Jarque, Carlos M. and Bera, Anil K. (1987), "A Test for Normality of Observations and Regression Residuals," *International Statistical Review* [@jarqueBera1987normality], the moment-based normality test combining squared skewness and excess-kurtosis contributions into a χ²(2) statistic.

**Why used.** NB2 §4 already executed both tests against `column6_fit.resid` *in memory* directly after the OLS fit. This §4 re-runs them against the residuals loaded from `env.FULL_PKL_PATH` — the pickle serialisation of the same fit object — so that the tests-and-sensitivity notebook verifies the handoff artifact carries residuals with the same diagnostic signature as NB2's in-memory reads. The re-run is therefore two things at once: (i) a final diagnostic pass on the primary specification's error structure, and (ii) a PKL round-trip integrity check — if statsmodels's pickle/unpickle path subtly altered residual values (floating-point accumulation order, for instance), the Q-statistic at lag 1 alone would move by several orders of magnitude at n=947 and the mismatch would surface here. We lock lags=1..8 for Ljung-Box per the plan's pre-committed range, matching the convention for weekly financial series (two months of calendar lag structure).

**Relevance to our results.** T4 (Ljung-Box) probes residual autocorrelation; T5 (Jarque-Bera) probes residual normality. Rejection at either test flags an OLS-assumption violation, but neither rejection invalidates the Column-6 primary estimate — they motivate the specific robustness machinery already layered into NB2. Ljung-Box Q rejections at short lags motivate the HAC(4) standard errors pre-registered for the primary β̂ (Newey-West with bandwidth 4 ≈ n^(1/4) ≈ 947^(1/4)); the coefficient point estimate is consistent under autocorrelated errors, only the SE needs correction. Jarque-Bera rejections motivate the NB2 §5 Student-t MLE robustness refit; under heavy tails the Gaussian-OLS likelihood is efficient but not optimal, and the Student-t refit provides an efficiency comparison that gets reported in the NB2 §11 handoff alongside the primary estimates. Both expected rejections are baked into the robustness suite — §4's job is to confirm the serialised residuals reproduce those expected rejections quantitatively.

**Connection to simulator.** The Abrigo simulator's CPI impulse-response calibration reads `column6_fit.params['cpi_surprise_ar1']` as its coefficient and its HAC(4) standard error from `nb2_params_point.json` as the uncertainty input for scenario generation. §4 confirms the uncertainty input is the right one: if Ljung-Box rejects (it does), the HAC(4) wrapper is necessary and the simulator's confidence band is the HAC-wrapped one, not the naive OLS band. If Jarque-Bera rejects (it does), the simulator carries a "Student-t residual tail" flag that tells downstream Monte-Carlo machinery to draw innovation shocks from the fitted Student-t location-scale rather than Gaussian — a one-line swap in the shock-draw function that materially widens extreme scenarios. §4's role is to make these flags observable on the serialised handoff rather than require a re-fit against the raw panel.


In [ ]:
# §4 T4 Ljung-Box Q(1..8) + T5 Jarque-Bera on the PKL-serialised
# primary residuals. Both tests consume `column6_fit.resid`; §1 loaded
# `column6_fit` from _pkl already.
#
# Gate: if pkl_degraded, skip — residuals cannot be trusted under
# version drift. Downstream §10 writes `t4_skipped = true, t5_skipped =
# true` to the gate verdict in that case.
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

if pkl_degraded:
    print("§4 SKIPPED — pkl_degraded = True (see §1 version-drift warnings).")
    _ljungbox_df = None
    _jb_result = None
else:
    _resid = column6_fit.resid

    # Ljung-Box Q(1..8). statsmodels returns a DataFrame indexed by lag
    # with columns lb_stat, lb_pvalue. Column naming is the canonical
    # statsmodels default since v0.12; we match on that name in tests.
    _ljungbox_df = acorr_ljungbox(_resid, lags=8)

    # Jarque-Bera. statsmodels' stattools.jarque_bera returns a 4-tuple
    # (JB, JBpv, skew, kurtosis). We keep the 4-tuple shape so the reader
    # sees skew + kurtosis alongside the test statistic.
    _jb_result = jarque_bera(_resid)
    _jb_stat, _jb_pvalue, _jb_skew, _jb_kurtosis = _jb_result

    print("NB3 §4 T4 Ljung-Box Q(1..8) on column6_fit residuals:")
    for _lag, _row in _ljungbox_df.iterrows():
        _lb_stat = float(_row["lb_stat"])
        _lb_p = float(_row["lb_pvalue"])
        _v = "REJECT" if _lb_p < 0.05 else "FAIL TO REJECT"
        print(f"  Q({_lag:>2}) = {_lb_stat:>10.4f}  p = {_lb_p:.3e}  {_v}")

    _jb_verdict = "REJECT" if float(_jb_pvalue) < 0.05 else "FAIL TO REJECT"
    print()
    print("NB3 §4 T5 Jarque-Bera on column6_fit residuals:")
    print(f"  statistic          = {float(_jb_stat):.4f}")
    print(f"  p-value            = {float(_jb_pvalue):.3e}")
    print(f"  skewness           = {float(_jb_skew):.4f}")
    print(f"  excess kurtosis    = {float(_jb_kurtosis):.4f}")
    print(f"  verdict at α=0.05  = {_jb_verdict}")


**Interpretation — §4.** T4 (Ljung-Box Q(1..8)) and T5 (Jarque-Bera) on the PKL-serialised `column6_fit.resid` both strongly **REJECT** at α=0.05, mirroring NB2 §4's in-memory diagnostic run — PKL round-trip integrity is confirmed.

Ljung-Box returns Q-statistics rising from Q(1) ≈ 155.7 (p ≈ 9.8e-36) to Q(8) ≈ 496.1 (p ≈ 4.9e-102). All eight p-values are effectively zero. The primary residual series carries strong, persistent serial correlation across all tested lags. This is not a surprise: `rv_cuberoot` is realized volatility, and even after the cube-root transform the series retains the vol-clustering signature of raw RV. The Column-6 primary does not model vol dynamics — it models the contemporaneous response of `rv_cuberoot` to `cpi_surprise_ar1` under a linear mean specification with exogenous controls. Residuals are expected to carry the clustering the mean specification doesn't absorb, and that clustering is exactly what motivates the HAC(4) wrapper pre-registered for the primary β̂ SE. The Ljung-Box REJECT is therefore the *justification* for the HAC standard errors, not a problem for them. The gate verdict writer in §10 records all eight p-values under `t4_ljungbox_pvalues` along with the aggregate verdict `t4_verdict = REJECT`.

Jarque-Bera returns stat ≈ 540.97 with p ≈ 3.4e-118 — a stronger rejection of normality than we have sample size to distinguish from "infinitely strong." Decomposing: skewness ≈ 0.959 (modest right-skew) and excess kurtosis ≈ 6.17 (substantial leptokurtosis, about 3.2× the Gaussian value of ≈3.0). The kurtosis dominates the statistic. This is the textbook high-frequency financial-residuals pattern: thick tails from vol spikes around unmodeled shocks (global risk-off episodes, political shocks, oil-correction weeks not captured by the `oil_return` control), mild right-skew from the occasional extreme-upside RV week. As with the Ljung-Box rejection, this result is not a primary-spec killer — it is the justification for the NB2 §5 Student-t MLE robustness refit. The gate writer records `t5_stat`, `t5_pvalue`, `t5_skew`, `t5_kurtosis`, and `t5_verdict = REJECT`; the Abrigo simulator's shock-draw machinery picks up a `residual_distribution = student_t` flag from that verdict block.

Both rejections also constitute the PKL-integrity proof. The Ljung-Box Q(8) p-value is proportional to exp(-Q(8)/2) at large Q, so a 1-part-in-10⁶ change in residual values from pickle/unpickle would shift Q(8) enough to move log₁₀(p) by a noticeable margin. We see the NB2 in-memory Q(8) to within tight numerical tolerance, confirming the serialisation did not introduce floating-point drift.
